In [4]:
import numpy as np
import matplotlib.pyplot as plt

from sko.GA import GA

from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling

In [5]:
def obj_fun_1(x):
    f1 = 0.8*(x[0]**2 + x[0] -  1)*x[2] + 0.12*x[0] + 2.16*x[0] - 0.12
    f2 = (1 + x[0]**2)*x[3] + 0.4*x[0]**2 - 1.6*x[0] - 0.4
    f3 = (1 + x[0]**2)*x[4] + x[0]**2 - 1
    f4 = (1 + x[0]**2)*x[5] + 0.8*(x[0]**2 + x[0] - 1)
    f5 = x[2]*x[6] - 0.02*x[5] - x[4] - x[2]*x[3] - 0.16*x[3]
    f6 = x[6]**2 - 2*x[3]*x[6] + x[5]**2 + x[3]**2 - x[1]**2
    f7 = x[7] - x[1]*x[2]
    f8 = 0.0476*x[2]*x[7]**12 + x[2] - 2.104

    return abs(np.array([f1, f2, f3, f4, f5, f6, f7, f8]))


def obj_fun_2(x):
    f1 = x[4] + x[3] - 1.803
    f2 = (x[1] + x[2])*x[4] + 6.19116*x[3] - 1.803*(1.497 + 0.035)
    f3 = x[5] + x[3] - 0.328
    f4 = 0.28801*x[5] - x[1]*x[2]*x[4]
    f5 = (-6.19116*x[0] + x[0]*x[2] + x[1]*x[4] - x[2]*x[4])*x[5] + x[0]*x[2]*x[4]
    f6 = 1.571*x[6] + x[3] - 1.803
    f7 = x[7] - 0.000856*x[6]**2
    f8 = (x[4] - x[0])*x[8] - x[0]*x[4]
    f9 = x[8] - 377*x[1]*x[7]

    return abs(np.array([f1, f2, f3, f4, f5, f6, f7, f8, f9]))

In [6]:
def pond_obj_1(x):
    objs = obj_fun_1(x[:8])

    zip_ob = zip(objs, x[8:])

    return abs(sum([f*w for f,w  in list(zip_ob)]))

def pond_obj_2(x):
    objs = obj_fun_2(x[:8])

    zip_ob = zip(objs, x[8:])

    return abs(sum([f*w for f,w  in list(zip_ob)]))

In [13]:
constraint_eq_1 = [
    lambda x: 0.8*(x[0]**2 + x[0] -  1)*x[2] + 0.12*x[0] + 2.16*x[0] - 0.12,
    lambda x: (1 + x[0]**2)*x[3] + 0.4*x[0]**2 - 1.6*x[0] - 0.4,
    lambda x: (1 + x[0]**2)*x[4] + x[0]**2 - 1,
    lambda x: (1 + x[0]**2)*x[5] + 0.8*(x[0]**2 + x[0] - 1),
    lambda x: x[2]*x[6] - 0.02*x[5] - x[4] - x[2]*x[3] - 0.16*x[3],
    lambda x: x[6]**2 - 2*x[3]*x[6] + x[5]**2 + x[3]**2 - x[1]**2,
    lambda x: x[7] - x[1]*x[2],
    lambda x: 0.0476*x[2]*x[7]**12 + x[2] - 2.104
]    

In [8]:
lb_1 = [-3, -1, -2, -1, -1, -0.5, -1.5, -1.5]
ub_1 = [1, 1, 2, 1, 1, 0.5, 1.5, 1.5]

lb_2 = [-0.5, -1, -1, -1, 1, -1, -1, 0, -1]
ub_2 = [0.5, 1, 1, 1, 2, 1, 1, 1, 1]

In [ ]:
def error_moo(result):
    return [sum(x ** 2) for x in result]

In [ ]:
class Problema1(ElementwiseProblem):

    def __init__(self):
        super().__init__(
            n_var=8,      
            n_obj=8,      
            xl=lb_1,
            xu=ub_1
        )

    def _evaluate(self, x, out, *args, **kwargs):

        out["F"] = obj_fun_1(x)

In [ ]:
class Problema2(ElementwiseProblem):

    def __init__(self):
        super().__init__(
            n_var=9,      
            n_obj=9,      
            xl=lb_2,
            xu=ub_2
        )

    def _evaluate(self, x, out, *args, **kwargs):

        out["F"] = obj_fun_2(x)

In [ ]:
problem1 = Problema1()
problem2 = Problema2()

algorithm = NSGA2(
    pop_size=100,

    sampling=FloatRandomSampling(),

    crossover=SBX(
        prob=0.6,
        eta=15
    ),

    mutation=PM(
        prob=1.0 / problem1.n_var,
        eta=20
    ),

    eliminate_duplicates=True
)

result1 = minimize(
    problem1,
    algorithm,
    termination=("n_gen", 1500),
    seed=42,
    verbose=False,
    save_history=True
)

result2 = minimize(
    problem2,
    algorithm,
    termination=("n_gen", 1500),
    seed=42,
    verbose=False
)

In [ ]:
ga_test = GA(
    func=pond_obj_1,
    n_dim=16,
    size_pop=100,
    max_iter=1000,
    lb=lb_1 + [-1]*8,
    ub=ub_1 + [1]*8
)

In [ ]:
for n in range(50):
    ga = GA(
    func=lambda x: obj_fun_1(x)[0],
    n_dim=8,
    size_pop=100,
    max_iter=1000,
    lb=lb_1,
    ub=ub_1,
    constraint_eq=constraint_eq_1[1:]
)
    
    best_x, best_y = ga.run()
    print(best_y)

TypeError: object of type 'function' has no len()